In [1]:

import numpy as np
import polars as pl
from aeon.datasets.tsc_datasets import univariate

from autotsc import utils

In [2]:
df = (
    pl.scan_parquet("s3://tsc-glue/performance/*.parquet")
    #.filter(pl.col("model") != "mixed-v2")
    #.filter(pl.col("model") != "mixed-v3")
    #.filter(pl.col("model") != "mixed-v4")  
    #.filter(pl.col("model") != "mixed")
    #.filter(pl.col("model") != "mixed-v4-ray-r-5")

).collect(engine="streaming")
df

/tmp/ipykernel_2985359/1781713861.py:9: UserWarning: '(default_)region' not set; polars will try to get it from bucket

Set the region manually to silence this warning.
  ).collect(engine="streaming")


dataset,model,run,resampled,test_accuracy
str,str,i64,bool,f64
"""SemgHandMovementCh2""","""mr-hydra""",300,false,0.771111
"""FaceFour""","""mr-hydra""",100,false,0.943182
"""NonInvasiveFetalECGThorax2""","""stacker-v4-r3""",200,false,0.970483
"""DistalPhalanxOutlineCorrect""","""quant""",500,false,0.786232
"""ACSF1""","""quant""",200,false,0.92
…,…,…,…,…
"""InsectEPGSmallTrain""","""rdst""",500,false,0.991968
"""FordB""","""mr-hydra""",200,false,0.835802
"""ScreenType""","""rstsf""",100,false,0.541333


In [3]:
df.group_by("model").agg(pl.len()).sort("len", descending=True)

model,len
str,u32
"""rdst""",640
"""rstsf""",640
"""mr-hydra""",640
"""quant""",640
"""stacker-v4-r1""",18
"""stacker-v4-r3""",14
"""catch22""",4
"""hivecotev2""",3


In [4]:
from aeon.visualisation import plot_critical_difference

In [5]:
v = df.pivot(
    on="model", values="test_accuracy", index="dataset", aggregate_function="mean"
).drop_nulls()
methods = df["model"].unique().to_list()
v

dataset,mr-hydra,stacker-v4-r3,quant,rdst,rstsf,catch22,stacker-v4-r1,hivecotev2
str,f64,f64,f64,f64,f64,f64,f64,f64


In [6]:
plot_critical_difference(v.select(methods).to_numpy(), methods)

/home/petelin/AutoTSC/.venv/lib/python3.13/site-packages/aeon/visualisation/results/_critical_difference.py:181: RuntimeWarning: Mean of empty slice.
  ordered_avg_ranks = ranks.mean(axis=0)
/home/petelin/AutoTSC/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


ZeroDivisionError: division by zero

In [ ]:
def dataset_stats():

    stats = []
    for dataset in univariate:
        X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
        stats.append(
            {
                "dataset": dataset,
                "n_train": X_train.shape[0],
                "n_test": X_test.shape[0],
                "n_classes": len(np.unique(y_train)),
                "series_length": X_train.shape[2],
            }
        )
    return pl.DataFrame(stats)


stats = dataset_stats()

In [ ]:
joined = v.join(stats, on="dataset").sort("n_train")
joined

In [ ]:
s_small = joined.filter(pl.col("n_train") < 200)
print(len(s_small))
plot_critical_difference(s_small.select(methods).to_numpy(), methods)

In [ ]:
s_medium = joined.filter(pl.col("n_train") >= 200).filter(pl.col("n_train") < 600)
print(len(s_medium))
plot_critical_difference(s_medium.select(methods).to_numpy(), methods)

In [ ]:
s_large = joined.filter(pl.col("n_train") >= 600)
print(len(s_large))
plot_critical_difference(s_large.select(methods).to_numpy(), methods)

In [ ]:
s_large

In [ ]:
import ray
ray.shutdown()